# Speech Denoising — Notebook 05: Inference & Evaluation

Goal: run the trained U-Net on a full raw test file (not just one chunk),
reconstruct the waveform, listen to the result, and score it with PESQ and STOI.

In [16]:
from model import UNet
from utils import slice_array, denoise_chunk
import torch
import librosa as lb
import numpy as np
import soundfile as sf
from IPython.display import Audio, display
from pesq import pesq
from pystoi import stoi

## Load Trained Model

10 epochs was chosen as the final checkpoint over 20 — see the overfitting note
further down.

In [17]:
model = UNet()
device = torch.device("mps")
model = model.to(device)
model.load_state_dict(torch.load("models/best_model.pth"))
model.eval();

## Load a Raw Test File

Unlike notebooks 01-04, this loads one full untouched .wav file from the test
split, not a preprocessed .npy chunk.

In [18]:
testfile_clean, clean_sr = lb.load("data/archive/clean_testset_wav/p232_363.wav", sr=None)
testfile_noisy, noisy_sr = lb.load("data/archive/noisy_testset_wav/p232_363.wav", sr=None)

## Slice Into Chunks

Same `slice_array` function used in notebook 02, now shared through `utils.py`.

In [19]:
CHUNK_SIZE = 16000
test_chunks_noisy = slice_array(testfile_noisy, CHUNK_SIZE)
len(test_chunks_noisy)

4

## Denoise Each Chunk

`denoise_chunk` (in `utils.py`) takes one raw noisy chunk, computes its STFT,
predicts the clean amplitude through the model, and also returns the noisy
phase — needed for reconstruction, since the model only ever predicts amplitude.

In [20]:
model.eval()
clear_list = []
phase_list = []
with torch.no_grad():
  for chunk in test_chunks_noisy:
    clear_chunk, stft_chunk_ph = denoise_chunk(chunk, model, device)
    clear_list.append(clear_chunk)
    phase_list.append(stft_chunk_ph)
print(len(clear_list), len(phase_list))

4 4


## Reconstruct the Waveform

Noisy phase approximation: combine the predicted amplitude with the noisy
chunk's own phase (`amplitude * exp(1j * phase)`), then inverse STFT back to
waveform. `length=16000` forces the exact chunk length back — `istft` doesn't
always return it automatically.

In [21]:
denoised_chunks = []
for amp, phase in zip(clear_list, phase_list):
  amp_np = amp.cpu().numpy().squeeze()
  complex_stft = amp_np * np.exp(1j * phase)
  wave_chunk = lb.istft(complex_stft, length=16000)
  denoised_chunks.append(wave_chunk)

In [22]:
denoised_wave = np.concatenate(denoised_chunks)
denoised_wave.shape

(64000,)

## Listen

Noisy input, our denoised output, and the true clean reference, side by side.

In [23]:
print("Noisy:")
display(Audio(testfile_noisy, rate=16000))
print("Denoised:")
display(Audio(denoised_wave, rate=16000))
print("Clean:")
display(Audio(testfile_clean, rate=16000))

Noisy:


Denoised:


Clean:


## PESQ & STOI Evaluation

`denoised_wave` is padded to a multiple of `CHUNK_SIZE` (last chunk was
zero-padded), so it's trimmed to the original file length before scoring.

In [24]:
denoised_wave_trimmed = denoised_wave[:len(testfile_clean)]

In [25]:
pesq_score = pesq(16000, testfile_clean, denoised_wave_trimmed)
stoi_score = stoi(testfile_clean, denoised_wave_trimmed, 16000)
print(f"PESQ: {pesq_score}")
print(f"STOI: {stoi_score}")

PESQ: 2.124760627746582
STOI: 0.9729783728135052


## Experiment: 10 vs 20 Epochs

Tried continuing training for 10 more epochs on top of the original checkpoint
(20 epochs total). Train loss kept improving (0.114 -> 0.089), but every
held-out metric got worse:

| Metric | 10 epochs | 20 epochs |
|---|---|---|
| Test Loss (MSE) | 0.045 | 0.054 |
| PESQ | 2.125 | 2.072 |
| STOI | 0.973 | 0.972 |

Train loss dropping while test loss rises is a textbook overfitting signal —
past a point the model was fitting the specific train chunks rather than the
general denoising pattern. The 10-epoch checkpoint (`models/best_model.pth`)
was kept as the final model.

## Save Audio Samples

Saved to `samples/` so the result can be heard directly from the GitHub repo,
without needing to run the notebook.

In [26]:
sf.write("samples/p232_363_noisy.wav", testfile_noisy, 16000)
sf.write("samples/p232_363_denoised.wav", denoised_wave_trimmed, 16000)
sf.write("samples/p232_363_clean.wav", testfile_clean, 16000)
print("Saved to samples/")

Saved to samples/
